# 09 differential results
Hypothesis classes first, then untargeted. Remember: per-animal means, paired by slide, n = 6 pairs.
`could_be_position` = same sign as and not clearly bigger than the top-to-bottom technical trend.

In [ ]:
# ---- setup: the same first cell in every review notebook ---------------------------------
# These notebooks only LOOK at outputs the scripts already wrote; they never recompute.
# Point CONFIG at the config you ran the pipeline with.
import sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from cln3msi.config import load

CONFIG = REPO / "config.yaml"          # or REPO / "config.test.yaml" for the synthetic run
ctx = load(CONFIG)
OUT = ctx.output_root
MODE = "lip_neg"
print("outputs in", OUT)

def show(*parts, width=900):
    p = OUT.joinpath(*parts)
    display(Image(filename=str(p), width=width)) if p.exists() else print("missing:", p)

def summary(name):
    p = OUT / "summaries" / name
    display(Markdown(p.read_text())) if p.exists() else print("missing:", p)

In [ ]:
R = pd.read_csv(OUT / "results" / "09_differential" / MODE / "differential.csv")
cols = ["unit", "label", "hypothesis", "log2fc", "ci_low", "ci_high", "p_ttest", "q_targeted", "q_all",
        "p_wilcoxon", "p_mixedlm", "n_pairs_up", "could_be_position", "ambiguous"]
R[R.targeted].sort_values("p_ttest")[cols].head(40)

In [ ]:
R[~R.targeted & (R.unit_kind == "region")].sort_values("p_ttest")[cols].head(30)

In [ ]:
show("figures", "09_differential", MODE, "volcano_regions.png", width=1100)
show("figures", "09_differential", MODE, "forest_hypothesis.png")

In [ ]:
# Per-animal values for one lipid in one region, paired by slide
m = pd.read_csv(OUT / "results" / "09_differential" / MODE / "per_animal_means.csv")
sheet = pd.read_csv(ctx.sample_sheet)
peak = R[R.targeted].sort_values("p_ttest").iloc[0]["peak"]
unit = "whole_section"
d = m[(m.peak == peak) & (m.unit == unit)].merge(sheet, on="animal_id")
fig, ax = plt.subplots(figsize=(4, 4))
for s, g in d.groupby("slide"):
    g = g.set_index("genotype").loc[["Control", "Mutant"]]
    ax.plot([0, 1], g["mean_norm_intensity"], marker="o", label=s)
ax.set_xticks([0, 1], ["Control", "CLN3 KO"]); ax.set_title(f"{peak} in {unit}"); ax.legend(title="slide")

In [ ]:
show("figures", "10_report", MODE, "top_hits_images.png", width=1100)